In [1]:
import pickle
import numpy as np

n_samples = 1000
n_feature = 100

np.random.seed(42)
X = np.random.normal(size=(n_samples, n_feature))
Signals = [(X[:, 0] ** 2), (np.log(np.abs(X[:, 1]) + 1)), (X[:, 2] * X[:, 3]), np.sin(X[:, 4])]
print('Effect size of each nonlinear component:', [Sig.std() for Sig in Signals])

for i, s in zip(range(4), [1,1,2,1]):
    Signals[i] = Signals[i]/(Signals[i].std()/s)
print('Effect size of each nonlinear component:', [Sig.std() for Sig in Signals])

y = Signals[0]+Signals[1]+Signals[2]+Signals[3]

ep = np.random.normal(scale = y.std()/10, size = n_samples)
y = y+ep
y = (y-y.mean())/y.std()

Effect size of each nonlinear component: [1.4284974856511834, 0.29972688335728004, 1.0403301129845928, 0.6619187606192499]
Effect size of each nonlinear component: [1.0, 1.0, 1.9999999999999998, 1.0]


In [2]:
dataset = {}
dataset['X'] = X
dataset['y'] = y
with open('simulation_data.pickle', 'wb') as f:
    pickle.dump(dataset, f)

In [3]:
def make_5_folds_from_array(arr, n_folds=5, seed=42):
    rng = np.random.default_rng(seed)
    rng.shuffle(arr)
    folds = np.array_split(arr, n_folds)
    return folds

rng = np.random.default_rng(42)

idx = make_5_folds_from_array(np.array(list(range(n_samples))))

test_idx = list(range(5))
val_idx = [[(i+1)%5] for i in range(5)]
train_idx = [[(i+j)%5 for j in range(2,5)] for i in range(5)]

fold_dic = {}

for k, test, val, train in zip(range(5), test_idx, val_idx, train_idx):
    dic = {}
    temp = idx[test].tolist()
    rng.shuffle(temp)
    dic['test']=np.array(temp)

    temp = []
    for i in val:
        temp+=idx[i].tolist()
    rng.shuffle(temp)
    dic['val']=np.array(temp)

    temp = []
    for i in train:
        temp+=idx[i].tolist()
    rng.shuffle(temp)
    dic['train']=np.array(temp)
    fold_dic[k] = dic

with open("5fold_setting.pickle","wb") as fw:
    pickle.dump(fold_dic, fw)